# 事前準備：共通コードの実行
* このノートブックに接続したら，まずは以下の2つの共通コード（コードAとコードB）を実行する
* これらの共通コードを実行しないと，それ以降のコードが実行できないので注意する
* また，コードA，コードB，及びコードC は，ノートブックに接続するたび毎回実行すること（ノートブックに接続中は，何度も実行する必要はない）
* 共通コードの詳細についての説明は割愛する（簡単な説明は[第2回のサンプルノートブック](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook02.ipynb)を参照）

In [ ]:
# コードA：日本語化ライブラリ導入
! pip install japanize-matplotlib | tail -n 1

In [ ]:
# コードB：共通事前処理

# B1:余分なワーニングを非表示にする
import warnings
warnings.filterwarnings('ignore')

# 必要ライブラリのimport
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib # matplotlib日本語化対応
import seaborn as sns

# B2:データフレーム表示用関数
from IPython.display import display

# B3:表示オプション調整
np.set_printoptions(suppress = True, precision = 3) #numpyの浮動小数点の表示精度
pd.options.display.float_format = '{:.3f}'.format #pandasでの浮動小数点の表示精度
pd.set_option('display.max_columns', None) #データフレームですべての列データを表示

# B4:グラフのデフォルトフォント指定
plt.rcParams['font.size'] = 14

# 乱数の種
random_seed = 1

In [ ]:
# コードC：データのダウンロード
!curl -L -o Boston.csv https://raw.githubusercontent.com/yoshida-nu/lecture_public/main/datascience/resources/08/Boston.csv

# 目的と到達目標

## 目的
* 機械学習におけるバイアスとバリアンスについて理解する
* リッジ回帰，ラッソ回帰，回帰木の特徴及び学習のしくみについて理解する


## 到達目標
* バイアスとバリアンスの観点で過学習が生じる原因を説明できる
* リッジ回帰，ラッソ回帰，及び回帰木のモデルの学習を実践できる


# バイアスとバリアンス

## 回帰で仮定しているデータ生成のメカニズム（母集団）
* 回帰におけるモデルの学習は観測したデータ（データセット）$𝐷=\{(𝑡_1, \bm{𝒙}_1), (𝑡_2, \bm{𝒙}_2), \ldots, (𝑡_𝑛, \bm{𝒙}_𝑛)\}$に基づいて行われる
* 機械学習ではデータセットが真のモデル $𝑓^∗(\bm{𝒙})$ から生成されると仮定している
* また，観測されるデータ $𝑡$ は真のモデルから生成されたものにノイズ $𝜀$ が加わっていると仮定している
$$
(𝑡_𝑖, \bm{𝒙}_𝑖)= (𝑓^∗ (\bm{𝒙}_𝑖 )+\varepsilon_𝑖, \bm{𝒙}_𝑖), \qquad 𝑖=1,2,\ldots,𝑛
$$
* ノイズ $\varepsilon$ は確率的に（ランダムに）決まる値と仮定
* したがって，データ $𝑡$ もランダムに決まる ⇒ 同じ $\bm{𝒙}$ に対して，観測ごとに異なる値をとる


### データ$t$を生成するメカニズム
* 回帰で仮定しているデータ生成のメカニズム: $𝑡 = 𝑓^∗ (\bm{𝒙}) + \varepsilon$
* ノイズ $\varepsilon$ は確率的に（ランダムに）決まるので，$𝑡$ は $\varepsilon$が従う確率分布に依存して生成される ⇒ $𝑡$ も確率分布に従う
* $\bm{𝒙}$が与えられたもとでのデータ $𝑡$ の確率分布を $𝑝(𝑡|\bm{𝒙})$ とおく
* $𝑝(𝑡|\bm{𝒙})$ は $𝑓^∗ (\bm{𝒙})$ を中心にばらつく（ばらつきは $\varepsilon$ が従う確率分布に依存）

  <img src="./fig/Bias-Variance_probdist.jpg" width="350">

### データセットと学習済みモデル
* したがって，データセットは観測するたび異なるものになる

  <img src="./fig/Bias-Variance_datagen_mechanism.jpg" width="550">

* また，モデルの学習はデータセットに依存する ⇒ データセットによって学習済みモデルが異なる

  <img src="./fig/Bias-Variance_model_learning.jpg" width="550">

### データセットとモデルの予測値
* 学習済みモデルが異なれば，予測値も異なる ⇒ データセットによって予測結果が異なる

  <img src="./fig/Bias-Variance_predict.jpg" width="500">

* 予測値は平均値（正確には期待値）を中心にばらつく

  <img src="./fig/Bias-Variance_predicted_value.jpg" width="500">

## バリアンスとは
* バリアンスとは予測値のモデルによるばらつきのこと
* 下図（上）は，バリアンスが小さいモデル．下図（下）は，バリアンスが大きいモデル
* つまり，選択する（仮定する）モデルによって，バリアンスの大きさが異なる
* 下図は，予測値の平均値が同じモデルになっているが，一般にはモデルによって予測値の平均値も異なる

  <img src="./fig/Bias-Variance_variance.jpg" width="500">

## バイアスとは
* バイアスとは予測値の平均値（期待値）と真の値 $𝑓^∗ (\bm{𝒙})$ との差のこと
* 下図（上）は，バイアスが小さいモデル．下図（下）は，バイアスが大きいモデル
* つまり，選択する（仮定する）モデルによって，バイアスの大きさが異なる
* 下図は，予測値の平均値が同じモデルになっているが，一般にはモデルによって予測値の平均値も異なる

  <img src="./fig/Bias-Variance_bias.jpg" width="580">

## バイアス-バリアンス分解
* 「バリアンス」とはモデルの予測値とその平均の誤差の平均
  * 正確にはモデルの予測値とその期待値との二乗誤差の期待値
* 「バイアス」とは真の値とモデルの予測値の期待値の誤差の平均
  * 正確には真の値とモデルの予測値の期待値との二乗誤差の期待値
* 「真の値と予測値の誤差（正確には二乗誤差の期待値）」は「バイアス」と「バリアンス」に分解できる
  * 真の値と予測値の誤差 ＝ バイアス ＋ バリアンス

## バイアスとバリアンスの観点でみる過学習
* モデルが訓練データの特徴を十分にとらえきれていない ⇒ バイアスが大きい状態になる
* モデルを複雑にすることでバイアスは小さくなる
  * 線形回帰であれば説明変数の数を増やす
  * 分類木であれば木の最大深さを大きくする
* しかし，モデルを複雑にすると，訓練データに過剰に当てはまったモデルを学習してしまう ⇒ 過学習が発生する
* 過学習はバイアスに対してバリアンスが大きい状態になっている
* 過学習が発生するしくみ
  * モデルを複雑にすると，（たまたま）観測したデータセットに対して当てはまりの良いモデルを学習してしまう
  * その結果，未知の値に対する予測結果にばらつきが生じる ⇒ バリアンスが大きい状態になる

  <img src="./fig/Bias-Variance_variance2.jpg" width="800">

  

## 過学習を発生させないためのアプローチ
* 過学習を起こさない ⇒ バリアンスを小くする
* バリアンスを小さく保ちつつ，バイアスも小さくしたい
* バリアンスを小くする2つのアプローチ
  * データ数を増やす
  * 異なるデータセットでも同じような学習結果になる機械学習手法（モデル）を選択する
* 回帰におけるモデル（回帰モデル）には，線形回帰モデル以外にも様々なモデルがある
* バリアンスを小さくするために適切なモデルを選択することも重要

# 代表的な回帰モデル
* 線形回帰
  * 最も基本的なモデルで説明変数と目的変数が線形関係にあると仮定
* 多項式回帰
  * 線形回帰を拡張し，説明変数の高次の項を含めたモデル
  * 線形回帰モデルの説明変数を2乗，3乗，… した変数や説明変数同士の積などを説明変数に含んだモデルとなる
  * より複雑な非線形関係をモデル化できる
* リッジ回帰／ラッソ回帰
  * 線形回帰モデルと多項式回帰モデルは，回帰係数を求めるために最小化する損失関数を平均二乗誤差としている
  * これに対して，リッジ回帰モデルやラッソ回帰モデルは，平均二乗誤差にに正則化項（後述）を加えたものを損失関数としている
  * 過学習を起こしにくいモデルとなる
* エラスティックネット（Elastic Net）
  * リッジ回帰とラッソ回帰の正則化項を組み合わせたモデル
* 回帰木
  * 分類木の概念を回帰に応用したモデル
  * 説明変数と目的変数の間の非線形関係もモデル化できる
  * そのため，複雑なデータ構造に対応する柔軟性を持つ
  * スケーリングの影響が少ないため，データの標準化を必要としない
  * 変数間の複雑な関係を木構造で可視化できるため，人間が理解しやすい形でモデルを示すことができる
* ランダムフォレスト
  * 複数の回帰木を組み合わせたモデル
  * 各回帰木の予測を平均化することで，モデルの精度を向上させる

## リッジ回帰／ラッソ回帰

### 回帰係数の大きさによる予測値への影響
* 説明変数が2つ以上（$𝑚$個）$\bm{x}=(𝑥_1, 𝑥_2, \ldots, x_m)$の線形回帰モデル:
$$
𝑓(\bm{x}) = 𝑎_1 𝑥_1 + 𝑎_2 𝑥_2 + \cdots + 𝑎_m 𝑥_m + 𝑏
$$
* 線形回帰モデルにおける回帰係数 $a_1, a_2, \ldots, a_m, b$ の決め方
  * $n$個の訓練データ（説明変数） $\bm{𝒙}_i, \ i=1,2,\ldots,n$ におけるモデルの値 $𝑓(\bm{x}_i)$ と正解データ $𝑡_i$ の差の2乗の平均値である平均2乗誤差関数を損失関数として定義
  $${\displaystyle \frac{1}{n} \sum_{i=1}^{n} \left( t_i - f(\bm{x}_i) \right)^2}$$
  * この損失関数が最小となる回帰係数を求めて（学習して）モデル $𝑓(𝑥)$ を作る
* 一般に回帰係数が大きいモデルは説明変数の値が少し変わっただけでも，目的変数の値が大きく変動する ⇒ 説明変数の値に敏感なモデルになる
* このようなモデルは，ノイズに敏感なモデルになり，予測が不安定になることが問題になる
* 例として，係数の異なる以下の2つの回帰モデルとその時の平均2条誤差関数の値を考える
  * モデル1: $𝑦 = 500 𝑥 − 20000$ ⇒ 平均2乗誤差関数: $50$
  * モデル2: $𝑦 = 100 𝑥 − 150$ ⇒ 平均2乗誤差関数: $60$
* 線形回帰モデルにおける回帰係数の決め方に従うと，平均2乗誤差が小さいモデル1のほうがよいモデルになる
* しかし，モデル1 は回帰係数が大きく，説明変数の値に敏感なモデルなので，予測が不安定になるかもしれない
* この点を踏まえると，平均2乗誤差が少し大きいけどモデル2のほうがよいともいえる？
* リッジ回帰やラッソ回帰は，モデル1のように回帰係数の大きいモデルではなく，モデル2のように回帰係数の小さいモデルを作る学習を行う


### 回帰係数の決め方の改良
* リッジ回帰やラッソ回帰では，回帰係数ができるだけ小さくなるように，平均2乗誤差関数に正則化項（または罰則項）と呼ばれる量を加えたものを損失関数とする（下式）
$$
\frac{1}{n}\sum_{i=1}^{n}\left(t_i-f(x_i)\right)^2
+\alpha\sum_{j=0}^{m}\left|a_j\right|^q
$$
* 正則化項は，係数が大きいとき，説明変数が多いときに大きな値をとる
* $\alpha$ は正則化パラメータと呼ばれる係数
* $𝑞=2$ ⇒ リッジ回帰モデル（正則化項は回帰係数の2乗の和）
* $𝑞=1$ ⇒ ラッソ回帰モデル（正則化項は回帰係数の絶対値の和）

## 回帰木
* 分類木を回帰に適用できる形にしたものが回帰木

### 回帰木モデルの例
* ここで下表のデータに対する回帰木モデルの学習を考える
  * 目的変数: 握力
  * 説明変数: 身長，体重

|**氏名**| **身長（cm）** | **体重（kg）** | **握力（kg）** |
|:--|:--|:--|:--|
|A| 170 | 60 | 40 |
|B| 158 | 40 | 29 |
|C| 175 | 68 | 38 |
|D| 160 | 53 | 31 |

* 学習した結果，下図の回帰木モデルが作成できたとする

  <img src="./fig/example_regression_tree.jpg" width="150">

* 回帰木では，葉ノード（下に分岐しないノード）に属するデータの平均値を予測値とする
* 例: 身長160cm，体重55kgの人の握力を予測する
  * BとDの握力の平均値 ⇒ 予測値: 30kg

### 回帰木の予測
* 分類木も回帰木も，各特徴量（説明変数）に対して，条件（分岐ルール）を定めてデータを2つに分割することを繰り返している
* 最終的に説明変数のデータを複数の領域に分割する
* 下図の例は特徴量が2つの例

  <img src="./fig/example_regression_tree2.jpg" width="300">

* 分類木では，各領域に1つのクラスを割り当て，その領域に入る説明変数に対する予測値としている
* 回帰木では，各領域に含まれるデータ（目的変数）の平均値を求め，その領域に入る説明変数に対するの予測値とする
* 回帰木の学習は，この領域を決めることに対応する

### 回帰木モデルの学習
* 分類木モデルも回帰木モデルも，不純度が小さくなるようにデータを複数の領域に分割（学習）する
* ただし，不純度の定義が異なる
* 分類木モデルの学習における不純度
  * 分岐後の領域に割り当てられたクラスと正解データが一致するデータの割合を示す量 ⇒ ジニ係数やエントロピーなどが使われる
* 回帰木モデルの学習における不純度
  * 分岐後の領域に属する訓練データ（目的変数）のばらつき具合を示す量 ⇒ 分散（尤離度）などが使われる

# 多項式回帰モデルによるデータ分析例：ボストン住宅データ

## 用いるデータと分析の目的
* 例として，csvファイル「Boston.csv」を用いる（各列名と意味は下表のとおり）
  * 出典: 須藤秋良, 株式会社フレアリンク: スッキリわかるPythonによる機械学習入門, インプレス, 2020
  * ここで利用するデータは，オリジナルのデータを加工して作成している
* ここでは，住宅価格を表すPRICEを予測する多項式回帰モデル（目的変数がPRICE）を作成することを考える

|**列名**| **意味** |
|:--|:--|
|CRIME| 犯罪率区分（high / low / very_low） |
|ZN| 広い住宅用地の割合 |
|INDUS| 非小売業用地の割合 |
|CHAS| チャールズ川の付近かどうかのダミー変数（1: 川周辺，0: それ以外） |
|NOX| 窒素酸化物濃度 |
|RM| 1戸当たりの平均部屋数 |
|AGE| 古い住宅の割合 |
|DIS| 雇用施設までの距離 |
|RAD| 幹線道路へのアクセス指数（アクセスのしやすさの指標） |
|TAX| 固定資産税率 |
|PTRATIO| 生徒・教師比率（教師1人当たりの生徒数） |
|LSTAT| 低所得者層の割合 |
|PRICE| その地域の住宅平均価格（1,000ドル単位） |

## データの前処理

### 前処理の概要
* 以下の前処理を順に行う
  * 学習に利用しないCRIME列を削除する（欠損値への対処を単純化するためでもある）
  * 欠損値を算術平均に置き換える
  * 外れ値を選択・削除する
  * 目的変数と説明変数に分割
  * 訓練データ（70％）とテストデータ（30%）に分割
  * データの標準化
* 本来は「訓練データ＆検証データ」と「テストデータ」に分割し，モデルをチューニングする作業も行うが，ここではチューニング作業を省略する

### 実習：列の削除／欠損値の対処／外れ値の削除
* `drop`メソッドを用いて，CRIME列をDataFrameから削除
  * `axis=1`でDataFrameの列データを削除
* 欠損値を算術平均に置き換える
  * `mean`メソッドで求めた算術平均を引数として，`fillna`メソッドを呼び出す
* `drop`メソッドで外れ値を削除
  * 前回と同様に，インデックス76のデータを外れ値として削除する
  * `axis=0`でDataFrameの行データを削除
  
**［実習内容］**
* 「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2行目: ファイルをDataFrameとして読み込み，変数`df`に代入
* 3行目: `drop`メソッドで，`df`からCRIME列を削除し，その結果を`df`に代入
* 4行目: `fillna`メソッドと`mean`メソッドを使って，`df`の欠損値を算術平均で穴埋めし，その結果を`df`に代入
* 5行目: `drop`メソッドで，`df`から外れ値（インデックス76に対応する行データ）を削除し，その結果を`df`に代入
* 6行目: `isnull`メソッドと`sum`メソッドを組み合わせて，各列の欠損値の数を計算して，`display`関数で表示する

**［実行結果］**

  <img src="./fig/exercise_polynomial_regression_boston_preprocessing1.jpg" width="120">

In [ ]:
# CRIME列の削除／欠損値の対処／外れ値の削除
df = pd.read_csv('Boston.csv')



display(pd.DataFrame(df.isnull().sum(), columns=['欠損値の数']))

### 目的変数と説明変数に分割
* `PRICE`を目的変数とする
* 説明変数として，`RM`, `LSTAT`, `PTRATIO`を選択する
* さらに，説明変数として選択した `RM`, `LSTAT`, `PTRATIO` から決まる以下の量を説明変数に加える
  * `RM^2`: `RM`の2乗
  * `LSTAT^2`：, `LSTAT`の2乗
  * `PTRATIO^2`： `PTRATIO`の2乗
  * `RM LSTAT`： `RM`と`LSTAT`の積
  * `RM PTRATIO`： `RM`と`PTRATIO`の積
  * `LSTAT PTRATIO`： `LSTAT`と`PTRATIO`の積
* `RM LSTAT`, `RM PTRATIO`, `LSTAT PTRATIO` を交互作用項と呼ぶ
* また，このモデルを次数が2（2次）の多項式回帰モデルと呼ぶ
* 任意の次数の多項式回帰モデルの説明変数は，`sklearn`（scikit-learn）の`preprocessing`モジュールにおける，`PolynomialFeatures`クラスを利用することで簡単に作成できる
  
**［`PolynomialFeatures`クラスによる説明変数の作成手順］**
* まず，多項式回帰モデルの説明変数を生成するためのオブジェクトを`PolynomialFeatures`クラスから生成し，変数に代入する
* 書式: `変数 = PolynomialFeatures(degree=次数, include_bias=False)`  
  * `include_bias`: 定数項を含む（`True`），含まない（`False`）を指定
  * `include_bias=True`にすると，すべての値が1の列（定数項）を作成する
  * `include_bias=False`は，この列を作成しないということになるが，モデルの学習において定数項を0にするという意味ではないことに注意する
* 生成したオブジェクトの`fit_transform`メソッドに説明変数を渡して呼び出す
* 書式: `変数.fit_transform(元の説明変数)`
  * 引数で指定した説明変数に対して，次数に応じた多項式回帰モデルの説明変数が生成される
  * 戻り値のクラス（データ型）は ndarray
  
**［以下のコードの処理内容］**
* 2行目: ファイルをDataFrameとして読み込み，変数`df`に代入
* 3～5行目: 列の削除，欠損値の対処，外れ値の削除
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 6行目: 説明変数の列名を要素とするリスト`['RM', 'PTRATIO', 'LSTAT']`を変数`x_cols`に代入
* 7行目: 目的変数の列名（リスト）`['PRICE']`を変数`t_col`に代入
* 8行目: DataFrame `df` から,`df[x_cols]`で，説明変数の列だけ取り出し変数`x`に代入
* 9行目: DataFrame `df` から,`df[t_col]`で目的変数の列だけ取り出し変数`t`に代入
* 10行目: `preprocessing`モジュールの`PolynomialFeatures`クラスのインポート
* 11行目: `PolynomialFeatures`クラスのオブジェクトを生成し，変数`pf`に代入
  * `degree=2`で，2次の多項式回帰モデルを指定
  * `include_bias=False`で，定数項の列を作成しない
* 12行目: `fit_transform`メソッドで，2次の多項式回帰モデルの説明変数データを生成し，変数`x`に代入
  * 引数は3つの説明変数 `RM`, `PTRATIO`, `LSTAT` のデータ `x`
  * このデータをもとに2次の多項式回帰モデルの説明変数データ（2次の項） `RM^2`, `LSTAT^2`, `PTRATIO^2`, `RM LSTAT`, `RM PTRATIO`, `LSTAT PTRATIO`を生成
* 13行目: `DataFrame`関数で，`x`をDataFrameに変換し，その結果を変数`x`に代入
  * `pf.get_feature_names_out()`で，2次の多項式回帰モデルの説明変数名を取り出し，DataFrameの列名（`columns`）として設定
* 14行目: `display`関数で，`x`の内容を表示

In [ ]:
# 目的変数と説明変数に分割
df = pd.read_csv('Boston.csv')



x_cols = ['RM', 'PTRATIO', 'LSTAT']
t_col = ['PRICE']
x = df[x_cols]
t = df[t_col]
from sklearn.preprocessing import PolynomialFeatures
pf = PolynomialFeatures(degree=2, include_bias=False)
x = pf.fit_transform(x)
x = pd.DataFrame(x, columns=pf.get_feature_names_out())
display(x)

### 実習：訓練データとテストデータに分割／データの標準化
* 説明変数と目的変数に分割できたので，次に，訓練データとテストデータに分割し，データの標準化を行う
* 分割と標準化の処理は第7回「[サンプルノートブック07](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook07.ipynb)」のコードと同様
  
**［実習内容］**
* 「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2行目: ファイルをDataFrameとして読み込み，変数`df`に代入
* 3～5行目: 列の削除，欠損値の対処，外れ値の削除（前の実習のコードと同じ）
* 6～13行目: 説明変数と目的変数に分割し，変数`x`と`t`にそれぞれ代入（前のコードと同じ）
* 14行目: `model_selection`モジュールの`train_test_split`関数のインポート
* 15行目: 説明変数と目的変数を訓練データ（70％）と検証データ（30％）に分割し，`x_train`, `x_test`, `t_train`, `t_test`にそれぞれ代入
* 16行目: `preprocessing`モジュールの`StandardScaler`クラスのインポート
* 17行目: `StandardScaler`クラスのオブジェクトを生成し，変数`sc_x`に代入
* 18行目: `sc_x`に対し，`fit`メソッドを適用する（引数は`x_train`）
* 19行目: `sc_x`に対する`transform`メソッドで`x_train`を標準化し，その結果を変数`x_train_s`に代入
* 20行目: `DataFrame`関数で，`x_train_s`をDataFrameに変換し，その結果を変数`x_train_s`に代入
* 21行目: `sc_x`に対する`transform`メソッドで`x_test`を標準化し，その結果を変数`x_test_s`に代入
* 22行目: `DataFrame`関数で，`x_test_s`をDataFrameに変換し，その結果を変数`x_test_s`に代入
* 23～30行目: `x_train_s`, `x_test_s`, `t_train`, `t_test`の内容をそれぞれ表示
  
**［実行結果］**

  <img src="./fig/exercise_polynomial_regression_boston_preprocessing3.jpg" width="650">

In [ ]:
# 訓練データとテストデータに分割／データの標準化
df = pd.read_csv('Boston.csv')



x_cols = ['RM', 'PTRATIO', 'LSTAT']
t_col = ['PRICE']
x = df[x_cols]
t = df[t_col]
from sklearn.preprocessing import PolynomialFeatures
pf = PolynomialFeatures(degree=2, include_bias=False)
x = pf.fit_transform(x)
x = pd.DataFrame(x, columns=pf.get_feature_names_out())






x_train_s = pd.DataFrame(x_train_s, columns=x_train.columns)

x_test_s = pd.DataFrame(x_test_s, columns=x_test.columns)
print('================ x_train_s ================')
display(x_train_s.head(2))
print('================ t_train ================')
display(t_train.head(2))
print('================ x_test_s ================')
display(x_test_s.head(2))
print('================ t_test ================')
display(t_test.head(2))

## 実習：多項式回帰モデルの学習と評価
* 前処理したボストン住宅データで多項式回帰モデルの学習と評価を行う
* 学習と評価の処理は第7回「[サンプルノートブック07](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook07.ipynb)」のコードと同様
* ただし，予測の処理は除く
* 実行結果の決定係数をみると，テストデータに対する決定係数は，訓練データに対する決定係数よりも小さくなっており，過学習が発生している
  * これを過学習と判断するかは分析者次第のところもある

**［実習内容］**
* 「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2行目: ファイルをDataFrameとして読み込み，変数`df`に代入
* 3～5行目: 列の削除，欠損値の対処，外れ値の削除
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 6～13行目: 説明変数と目的変数に分割し，変数`x`と`t`にそれぞれ代入
* 14～15行目:  説明変数と目的変数を訓練データ（70％）と検証データ（30％）に分割し，`x_train`, `x_test`, `t_train`, `t_test`にそれぞれ代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 16～22行目: データの標準化を行い，その結果を`x_train_s`, `x_test_s`にそれぞれ代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 23行目:  `sklearn` (scikit-learn) の`linear_model`モジュール内にある`LinearRegression`クラスを読み込む 
* 24行目: `LinearRegression()`で，モデルの学習を行うためのオブジェクトを`LinearRegression`クラスから生成し，変数`linear_regression_model`に代入
* 25行目: 標準化した訓練データ（`x_train_s`と`t_train`）を用いて，`fit`メソッドでモデルの学習を実行
* 26行目: `score`メソッドで，標準化した訓練データに対する決定係数を計算して変数`score_train`に代入
* 27行目: `score`メソッドで，標準化したテストデータに対する決定係数を計算して変数`score_test`に代入
* 28行目: `print`関数とf-stringを使って，訓練データに対する決定係数（`score_train`）とテストデータに対する決定係数（`score_test`）を小数点以下3桁まで表示
* 29行目: `coef_`属性から回帰係数（定数項以外）を取り出し，変数`a`に代入
* 30行目: `intercept_`属性から定数項を取り出し，変数`b`に代入
* 31行目: `print`関数とf-stringを使って，回帰係数`a`を表示
* 32行目: `print`関数とf-stringを使って，定数項`b`を表示（標準化しているので 0 になる）

**［実行結果］**
```
訓練データの決定係数=0.852 / テストデータの決定係数=0.727
回帰係数（定数項以外）: [[ -9.002  16.951  14.872  18.916  -3.621 -20.468 -15.269   4.304   0.065]]
定数項: [22.387]
```

In [ ]:
# 多項式回帰モデルの学習と評価
df = pd.read_csv('Boston.csv')



x_cols = ['RM', 'PTRATIO', 'LSTAT']
t_col = ['PRICE']
x = df[x_cols]
t = df[t_col]
from sklearn.preprocessing import PolynomialFeatures
pf = PolynomialFeatures(degree=2, include_bias=False)
x = pf.fit_transform(x)
x = pd.DataFrame(x, columns=pf.get_feature_names_out())






x_train_s = pd.DataFrame(x_train_s, columns=x_train.columns)

x_test_s = pd.DataFrame(x_test_s, columns=x_test.columns)





print(f'訓練データの決定係数={score_train:.3f} / テストデータの決定係数={score_test:.3f}')



print(f'回帰係数（定数項以外）: {a}')
print(f'定数項: {b}')

# リッジ回帰モデルによるデータ分析例：ボストン住宅データ
* 多項式回帰モデルの学習を行った結果，過学習が起きていることが確認できた
* この問題を改善するためにモデルを変更する
* ここでは，リッジ回帰モデルで学習を試みる

## モデルの学習と評価
* ここでは，正則化パラメータを5としたときの，リッジ回帰モデルの学習を行う
* リッジ回帰モデルの学習は，`sklearn`（scikit-learn）の`linear_model`モジュールにおける`Ridge`クラスを使う
* `Ridge`クラスは，線形回帰モデルの学習を行うときに用いるクラス（`LinearRegression`クラス）とは異なるクラス
* リッジ回帰モデルの学習の流れ:
  * `Ridge`クラスを読み込む（インポートする）
  * インポートの書式: `from sklearn.linear_model import Ridge`
  * `Ridge`クラスのオブジェクトを生成し，変数に代入する
  * オブジェクト生成の書式: `変数 = Ridge(alpha=正則化パラメータ)`
  * モデルを学習する
  * モデルの学習の書式（これまでのモデルと同様）: `変数.fit(説明変数, 目的変数)`
* 以下のコードで，モデルの学習と評価（訓練データとテストデータに対する決定係数をそれぞれ計算）を行う
* 決定係数の計算は，これまでのモデルと同様に`score`メソッドを用いる
* 実行結果の決定係数から，過学習の問題が改善されていることがわかる
* ただし，正則化パラメータの値やデータによってモデルの性能（決定係数）が変わってくることに注意する
* 以下のコードでは，正則化パラメータを「5」としているが，本来はモデルの性能が向上するように調整する必要がある
  
**［以下のコードの処理内容］**
* 2行目: ファイルをDataFrameとして読み込み，変数`df`に代入
* 3～5行目: 列の削除，欠損値の対処，外れ値の削除
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 6～13行目: 説明変数と目的変数に分割し，変数`x`と`t`にそれぞれ代入
* 14～15行目:  説明変数と目的変数を訓練データ（70％）と検証データ（30％）に分割し，`x_train`, `x_test`, `t_train`, `t_test`にそれぞれ代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 16～22行目: データの標準化を行い，その結果を`x_train_s`, `x_test_s`にそれぞれ代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 23行目:  `sklearn` (scikit-learn) の`linear_model`モジュール内にある`Ridge`クラスを読み込む 
* 24行目: `Ridge(alpha=5)`で，モデルの学習を行うためのオブジェクトを`Ridge`クラスから生成し，変数`ridge_regression_model`に代入
  * `alpha=5`で，正則化パラメータを 5と設定
* 25行目: 標準化した訓練データ（`x_train_s`と`t_train_s`）を用いて，`fit`メソッドでモデルの学習を実行
* 26行目: `score`メソッドで，標準化した訓練データに対する決定係数を計算して変数`score_train`に代入
* 27行目: `score`メソッドで，標準化したテストデータに対する決定係数を計算して変数`score_test`に代入
* 28行目: `print`関数とf-stringを使って，訓練データに対する決定係数（`score_train`）とテストデータに対する決定係数（`score_test`）を小数点以下3桁まで表示
* 29行目: `coef_`属性から回帰係数（定数項以外）を取り出し，変数`a`に代入
* 30行目: `intercept_`属性から定数項を取り出し，変数`b`に代入
* 31行目: `print`関数とf-stringを使って，回帰係数`a`を表示
* 32行目: `print`関数とf-stringを使って，定数項`b`を表示（標準化しているので 0 になる）

In [ ]:
# リッジ回帰モデルの学習と評価
df = pd.read_csv('Boston.csv')



x_cols = ['RM', 'PTRATIO', 'LSTAT']
t_col = ['PRICE']
x = df[x_cols]
t = df[t_col]
from sklearn.preprocessing import PolynomialFeatures
pf = PolynomialFeatures(degree=2, include_bias=False)
x = pf.fit_transform(x)
x = pd.DataFrame(x, columns=pf.get_feature_names_out())






x_train_s = pd.DataFrame(x_train_s, columns=x_train.columns)

x_test_s = pd.DataFrame(x_test_s, columns=x_test.columns)
from sklearn.linear_model import Ridge
ridge_regression_model = Ridge(alpha=5)
ridge_regression_model.fit(X=x_train_s, y=t_train)
score_train = ridge_regression_model.score(X=x_train_s, y=t_train)
score_test = ridge_regression_model.score(X=x_test_s, y=t_test)
print(f'訓練データの決定係数={score_train:.3f} / テストデータの決定係数={score_test:.3f}')
a = ridge_regression_model.coef_
b = ridge_regression_model.intercept_
print(f'回帰係数（定数項以外）: {a}')
print(f'定数項: {b}')

## 回帰係数の大きさの比較
* 次に，多項式回帰モデルとリッジ回帰モデルの大きさを比較する
* 回帰係数（切片以外）は`coef_`プロパティ，切片は`intercept_`プロパティで確認できる
* 以下のコードでは，多項式回帰モデルとリッジ回帰モデル（正則化パラメータは5）の学習をそれぞれ行い，回帰係数の絶対値の合計をそれぞれ計算している
* なお，データの標準化により，定数項は 0 となるので，合計の計算には定数項を入れていない
  
**［以下のコードの処理内容］**
* 2行目: ファイルをDataFrameとして読み込み，変数`df`に代入
* 3～5行目: 列の削除，欠損値の対処，外れ値の削除
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 6～13行目: 説明変数と目的変数に分割し，変数`x`と`t`にそれぞれ代入
* 14～15行目:  説明変数と目的変数を訓練データ（70％）と検証データ（30％）に分割し，`x_train`, `x_test`, `t_train`, `t_test`にそれぞれ代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 16～22行目: データの標準化を行い，その結果を`x_train_s`, `x_test_s`にそれぞれ代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 23～27行目: 多項式回帰モデルの学習を行い，回帰係数を表示
* 28～32行目: リッジ回帰モデルの学習を行い，回帰係数を表示
* 33行目: `sum`関数と`abs`関数を使って，多項式回帰モデルの回帰係数の絶対値の合計を計算し，変数`sum_coef_linear`に代入
  *  `abs`関数は引数で指定した数値の絶対値を返す
  *  `coef_[0]`で，回帰係数のリストを取り出している
  *  `sum`関数は引数で指定した複数の数値（リストの要素）の合計を返す
* 34行目: 同様に `sum`関数と`abs`関数を使って，リッジ回帰モデルの回帰係数の絶対値の合計を計算し，変数`sum_coef_ridge`に代入
  *  `coef_`のデータ構造が，`linear_regression_model`と異なるので，回帰係数の指定が`coef_[0]`ではなく，`coef_`となっている
* 35行目: `print`関数とf-stringを使って，多項式回帰モデルの回帰係数の絶対値の合計 `sum_coef_linear`を表示
* 36行目: `print`関数とf-stringを使って，リッジ回帰モデルの回帰係数の絶対値の合計 `sum_coef_ridge`を表示

In [ ]:
# 多項式回帰モデルとリッジ回帰モデルの回帰係数の比較
df = pd.read_csv('Boston.csv')



x_cols = ['RM', 'PTRATIO', 'LSTAT']
t_col = ['PRICE']
x = df[x_cols]
t = df[t_col]
from sklearn.preprocessing import PolynomialFeatures
pf = PolynomialFeatures(degree=2, include_bias=False)
x = pf.fit_transform(x)
x = pd.DataFrame(x, columns=pf.get_feature_names_out())






x_train_s = pd.DataFrame(x_train_s, columns=x_train.columns)

x_test_s = pd.DataFrame(x_test_s, columns=x_test.columns)
from sklearn.linear_model import LinearRegression
linear_regression_model = LinearRegression()
linear_regression_model.fit(X=x_train_s, y=t_train)
a = linear_regression_model.coef_
print(f'多項式回帰モデルの回帰係数（定数項以外）: {a}')
from sklearn.linear_model import Ridge
ridge_regression_model = Ridge(alpha=5)
ridge_regression_model.fit(X=x_train_s, y=t_train)
a = ridge_regression_model.coef_
print(f'リッジ回帰モデルの回帰係数（定数項以外）: {a}')
sum_coef_linear = sum(abs(linear_regression_model.coef_[0]))
sum_coef_ridge = sum(abs(ridge_regression_model.coef_))
print(f'多項式回帰モデルの回帰係数の合計: {sum_coef_linear:.3f}')
print(f'リッジ回帰モデルの回帰係数の合計: {sum_coef_ridge:.3f}')

# ラッソ回帰モデルによるデータ分析例：ボストン住宅データ

## 実習：モデルの学習と評価
* ここでは，正則化パラメータを0.05としたときの，ラッソ回帰モデルの学習を行う
* ラッソ回帰モデルの学習は，`sklearn`（scikit-learn）の`linear_model`モジュールにおける`Lasso`クラスを使う
* 学習の流れは，リッジ回帰モデルと同様
* ラッソ回帰モデルの学習の流れ:
  * `Lasso`クラスを読み込む（インポートする）
  * インポートの書式: `from sklearn.linear_model import Lasso`
  * `Lasso`クラスのオブジェクトを生成し，変数に代入する
  * オブジェクト生成の書式: `変数 = Lasso(alpha=正則化パラメータ)`
  * モデルを学習する
  * モデルの学習の書式（これまでのモデルと同様）: `変数.fit(説明変数, 目的変数)`
* 以下のコードで，モデルの学習と評価（訓練データとテストデータに対する決定係数をそれぞれ計算）を行う
* 決定係数の計算は，これまでのモデルと同様に`score`メソッドを用いる
* リッジ回帰モデルの学習と同様，正則化パラメータの値やデータによってモデルの性能（決定係数）が変わってくることに注意する
* 以下のコードでは，正則化パラメータを「0.05」としているが，本来はモデルの性能が向上するように調整する必要がある

**［実習内容］**
* 「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2行目: ファイルをDataFrameとして読み込み，変数`df`に代入
* 3～5行目: 列の削除，欠損値の対処，外れ値の削除
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 6～13行目: 説明変数と目的変数に分割し，変数`x`と`t`にそれぞれ代入
* 14～15行目:  説明変数と目的変数を訓練データ（70％）と検証データ（30％）に分割し，`x_train`, `x_test`, `t_train`, `t_test`にそれぞれ代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 16～22行目: データの標準化を行い，その結果を`x_train_s`, `x_test_s`にそれぞれ代入
  * 前の実習のコードと同じなので，自分で入力する（コピペでよい）
* 23行目:  `sklearn` (scikit-learn) の`linear_model`モジュール内にある`Lasso`クラスを読み込む 
* 24行目: `Lasso(alpha=0.05)`で，モデルの学習を行うためのオブジェクトを`Lasso`クラスから生成し，変数`lasso_regression_model`に代入
  * `alpha=0.05`で，正則化パラメータを 0.05 と設定
* 25行目: 標準化した訓練データ（`x_train_s`と`t_train`）を用いて，`fit`メソッドでモデルの学習を実行
* 26行目: `score`メソッドで，標準化した訓練データに対する決定係数を計算して変数`score_train`に代入
* 27行目: `score`メソッドで，標準化したテストデータに対する決定係数を計算して変数`score_test`に代入
* 28行目: `print`関数とf-stringを使って，訓練データに対する決定係数（`score_train`）とテストデータに対する決定係数（`score_test`）を小数点以下3桁まで表示
* 29行目: `coef_`属性から回帰係数（定数項以外）を取り出し，変数`a`に代入
* 30行目: `intercept_`属性から定数項を取り出し，変数`b`に代入
* 31行目: `print`関数とf-stringを使って，回帰係数`a`を表示
* 32行目: `print`関数とf-stringを使って，定数項`b`を表示（標準化しているので 0 になる）

**［実行結果］**
```
訓練データの決定係数=0.822 / テストデータの決定係数=0.817
回帰係数（定数項以外）: [-0.     4.199  0.     9.946 -6.336 -7.237  0.     0.547  4.256]
定数項: [22.387]
```

In [ ]:
# ラッソ回帰モデルの学習と評価
df = pd.read_csv('Boston.csv')



x_cols = ['RM', 'PTRATIO', 'LSTAT']
t_col = ['PRICE']
x = df[x_cols]
t = df[t_col]
from sklearn.preprocessing import PolynomialFeatures
pf = PolynomialFeatures(degree=2, include_bias=False)
x = pf.fit_transform(x)
x = pd.DataFrame(x, columns=pf.get_feature_names_out())






x_train_s = pd.DataFrame(x_train_s, columns=x_train.columns)

x_test_s = pd.DataFrame(x_test_s, columns=x_test.columns)





print(f'訓練データの決定係数={score_train:.3f} / テストデータの決定係数={score_test:.3f}')


print(f'回帰係数（定数項以外）: {a}')
print(f'定数項: {b}')

## ラッソ回帰モデルの回帰係数について
* ラッソ回帰モデルの学習では，目的変数に影響を与えない説明変数の回帰係数を0にするという特徴があるが，実行結果から．それが確認できる

|  | 回帰係数 |
|---|---:|
| RM | 0 |
| PTRATIO | 4.199 |
| LSTAT | 0 |
| RM^2 | 9.946 |
| RM PTRATIO | -6.336 |
| RM LSTAT | -7.237 |
| PTRATIO^2 | 0 |
| PTRATIO LSTAT | 0.547 |
| LSTAT^2 | 4.256 |

# 回帰木モデルの学習

## データの前処理
* ここでは，回帰木モデルの学習を行う
* 説明変数は，目的変数である`PRICE`を除くすべての特徴量とする
* また，回帰木モデルは，データのスケールや外れ値の影響を受けない（受けにくい）ので，データの標準化と外れ値の対処は行わず学習する
* この特徴は分類木モデルも同様

  
**［以下のコードの処理内容］**
* 2行目: ファイルをDataFrameとして読み込み，変数`df`に代入
* 3～4行目: 列の削除，欠損値の対処
* 5～7行目: 説明変数と目的変数に分割し，変数`x`と`t`にそれぞれ代入
  * `df.drop(['PRICE'], axis=1)`で，`PRICE`列を削除したDataFrame（これが説明変数となる）を変数`x`に代入している
* 8～9行目: 説明変数と目的変数を訓練データ（70％）と検証データ（30％）に分割し，`x_train`, `x_test`, `t_train`, `t_test`にそれぞれ代入
* 10～17行目: `x_train`, `x_test`, `t_train`, `t_test`をそれぞれ表示

In [ ]:
# データの前処理
df = pd.read_csv('Boston.csv')
df = df.drop(['CRIME'], axis=1)
df = df.fillna(df.mean())
t_col = ['PRICE']
x = df.drop(['PRICE'], axis=1)
t = df[t_col]
from sklearn.model_selection import train_test_split
x_train, x_test, t_train, t_test = train_test_split(x, t, test_size=0.3, random_state=random_seed)
print('================ x_train ================')
display(x_train)
print('================ t_train ================')
display(pd.DataFrame(t_train))
print('================ x_test ================')
display(x_test)
print('================ t_test ================')
display(pd.DataFrame(t_test))

## モデルの学習と評価
* 回帰木モデルの学習は，`sklearn`（scikit-learn）の`tree`モジュールにおける`DecisionTreeRegressor`クラスを使う
* 回帰木モデルの学習の流れ:
  * `DecisionTreeRegressor`クラスを読み込む（インポートする）
  * インポートの書式: `from sklearn.tree import DecisionTreeRegressor`
  * `DecisionTreeRegressor`クラスのオブジェクトを生成（引数は分類木と同様）し，変数に代入する
  * オブジェクト生成の書式: `変数 = DecisionTreeRegressor(max_depth=[最大深さ], random_state=[乱数の種])`
  * モデルを学習する
  * モデルの学習の書式（これまでのモデルと同様）: `変数.fit(説明変数, 目的変数)`
* 以下のコードで，木の最大深さを4とした回帰木モデルの学習と評価（決定係数と特徴量重要度を計算）を行う
* 内容は第5回「[サンプルノートブック05](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook05.ipynb)」の一連の分析と同様（ただし，ここでは予測を省いている）
* さらに，`tree`モジュールの`plot_tree`関数で回帰木モデルを描画する（分類木モデルと同様）
* 学習と評価を行った結果，良いモデルはできなかったので，さらなるチューニングが必要であるが，今回はここまでとする
  
**［以下のコードの処理内容］**
* 2行目: ファイルをDataFrameとして読み込み，変数`df`に代入
* 3～4行目: 列の削除，欠損値の対処
* 5～7行目: 説明変数と目的変数に分割し，変数`x`と`t`にそれぞれ代入
* 8～9行目: 説明変数と目的変数を訓練データ（70％）と検証データ（30％）に分割し，`x_train`, `x_test`, `t_train`, `t_test`にそれぞれ代入
* 10行目: `sklearn` (scikit-learn) の`tree`モジュールを読み込む 
* 11行目: `tree.DecisionTreeClassifier`で，分類木モデルの学習を行うためのオブジェクトを`tree`モジュールの`DecisionTreeClassifier`クラスから生成し，変数`model_tree`に代入
* 12行目: 訓練データ（`x_train`と`t_train`）を用いて，`fit`メソッドでモデルの学習を実行
* 13行目: `model_tree.score(X=x_train, y=t_train)`で，訓練データに対する決定係数を計算して変数`score_train`に代入
* 14行目: `model_tree.score(X=x_test, y=t_test)`で，テストデータに対する決定係数を計算して変数`score_test`に代入
* 15行目: `print`関数とf-stringを使って，訓練データに対する決定係数（`score_train`）とテストデータに対する決定係数（`score_test`）を小数点以下3桁まで表示
* 16行目: `display`関数で，`model_tree`オブジェクトの`feature_importances_`属性（特徴量重要度）を表示
* 17行目: `pyplot`モジュールの`figure`関数を使って，図のサイズ（高さと幅）を指定
* 18行目: `plot_tree`関数で回帰木を描画
* 19行目: `show`関数で，それまでに設定した図（回帰木モデル）を実行画面に表示

In [ ]:
# 回帰木モデルの学習と評価
df = pd.read_csv('Boston.csv')
df = df.drop(['CRIME'], axis=1)
df = df.fillna(df.mean())
t_col = ['PRICE']
x = df.drop(['PRICE'], axis=1)
t = df[t_col]
from sklearn.model_selection import train_test_split
x_train, x_test, t_train, t_test = train_test_split(x, t, test_size=0.3, random_state=random_seed)
from sklearn import tree
model_tree = tree.DecisionTreeRegressor(max_depth=3, random_state=random_seed)
model_tree.fit(X=x_train, y=t_train)
score_train = model_tree.score(X=x_train, y=t_train)
score_test = model_tree.score(X=x_test, y=t_test)
print(f'訓練データの決定係数={score_train:.3f} / テストデータの決定係数={score_test:.3f}')
display(pd.DataFrame(model_tree.feature_importances_, index=x.columns, columns=['特徴量重要度']))
plt.figure(figsize=(15, 5))
tree.plot_tree(decision_tree=model_tree, feature_names=list(x.columns))
plt.show()